# Lesson 2.5 - File I/O and Error Handling

**Objectives**

- Read from and write to text and CSV files with Python's built-ins
- Explain what exceptions are and why programs raise them
- Handle errors gracefully with `try` / `except` / `finally`

This notebook actually creates files on disk as you run it. To keep the project folder tidy, everything gets written into a `scratch/` subfolder next to this notebook - that folder is just for this lesson's exercise output, not part of the course materials themselves.

In [ ]:
import os

os.makedirs("scratch", exist_ok=True)
print("scratch/ is ready")

## 1. Reading and writing text files

File `I/O` ("input/output") means reading data from files on disk, or writing data out to them. Python's built-in `open()` function is the standard way to do both.

The safest way to work with a file is with a `with` statement, which automatically closes the file for you when you're done — even if something goes wrong partway through:

In [ ]:
with open("scratch/notes.txt", "w") as file:
    file.write("Hello, file!\n")
    file.write("This is line two.\n")

print("wrote notes.txt")

`"w"` here is the mode — `"w"` means "write" (create the file if it doesn't exist, and overwrite it completely if it does). `as file` gives you a variable, `file`, representing the open file while you're inside the `with` block.

Reading it back uses `"r"` mode ("read"):

In [ ]:
with open("scratch/notes.txt", "r") as file:
    contents = file.read()

print(contents)

To add to a file  without erasing what's already there, use mode `"a"` ("append"):

In [ ]:
# "a" (append) adds to the file WITHOUT erasing what's already there
with open("scratch/notes.txt", "a") as file:
    file.write("This is line three.\n")


You can also loop over a file line by line, which is memory-efficient for large files:

In [ ]:
with open("scratch/notes.txt", "r") as file:
    for line in file:
        print(line.strip())   # .strip() removes the trailing newline

## 2. Reading and writing CSV files

A **CSV** file ("comma-separated values") is a plain text file storing `tabular data` — `rows` and `columns` — with commas separating the values in each row. Python's built-in `csv` module handles the fiddly details (like values that themselves contain commas) for you:

In [ ]:
import csv

rows = [
    ["name", "score"],
    ["Ada", 95],
    ["Grace", 88],
]

with open("scratch/scores.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

print("wrote scores.csv")

The `newline=""` argument avoids extra blank lines being inserted on some systems

Reading it back with `csv.DictReader` gives you each row as a `dictionary` keyed by the `header` row, which is usually the most convenient form to work with:

In [ ]:
with open("scratch/scores.csv", "r", newline="") as file:
    reader = csv.DictReader(file)
    for row in reader:
        print(row)

Notice the scores came back as **strings** (`'95'`, not `95`) - every value read from a CSV file **starts out as text**. You'd need `int(row["score"])` to do arithmetic with it. In Module 3, `pandas` handles this conversion for you automatically.

## 3. What is an exception?

An exception is Python's way of signaling that something went wrong while your program was running — a file that doesn't exist, a division by zero, a type mismatch. By default, an exception stops your program and prints a traceback, a message showing what went wrong and where.

In [ ]:
print(10 / 0)  # program crashes

print("following code after crashing")

Exceptions exist so that problems are reported loudly and immediately, rather than letting your program silently produce wrong answers.

## 4. Handling errors with `try` / `except`

Sometimes you expect an operation might fail, and you want your program to recover gracefully instead of crashing. That's what `try` / `except` is for:

In [ ]:
try:
    print(10 / 0)
except ZeroDivisionError as error:  # program catches error and avoid crashing
    print(f"Caught an error: {error}")
    print("Can't divide by zero!")

print("following code after try/except")

Python runs the `try` block; if an exception matching except `ZeroDivisionError` occurs, it jumps straight to that block instead of crashing, and the program continues normally afterward.

You can catch `multiple exception` types, and access the error message itself:

In [ ]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("Error: cannot divide by zero.")
        return None
    except TypeError as error:
        print(f"Error: {error}")
        return None

print(safe_divide(10, 2))
print(safe_divide(10, 0))
print(safe_divide(10, "2"))

## 5. `try` / `except` / `finally`

`finally` adds a block that runs **no matter what** — whether the try block succeeded or an exception was caught — which is useful for cleanup steps like closing a file:

In [ ]:
try:
    file = open("scratch/notes.txt", "r")
    contents = file.read()
finally:
    file.close()
    print("File closed.")

In practice, the `with` statement from earlier in this lesson already handles closing files for you automatically, so you'll reach for explicit `finally` blocks less often than `try/except` — but it's important to know it exists, since you'll see it in other people's code.

A very common real pattern: trying to open a file that might not exist.

In [ ]:
try:
    with open("scratch/does_not_exist.txt", "r") as file:
        contents = file.read()
except FileNotFoundError:
    print("That file doesn't exist yet.")

## Try it yourself

**Exercise 1.** Write a list of 3 short strings representing a to-do list. Write them to `scratch/todo.txt`, one per line (use a `for` loop and `file.write(item + "\n")`).

In [ ]:
# TODO: write a 3-item to-do list to scratch/todo.txt, one item per line



**Exercise 2.** Read `scratch/todo.txt` back and print each line with its line number, like `"1. Buy milk"` (use `enumerate(file, start=1)`).

In [ ]:
# TODO: read scratch/todo.txt and print each line with its number



**Exercise 3.** Write a function `safe_get_item(items, index)` that returns `items[index]`, but catches `IndexError` and returns `None` instead of crashing. Test it with a valid index and an out-of-range index.

In [ ]:
# TODO: define safe_get_item(items, index) with try/except IndexError



### Solution

In [ ]:
# Exercise 1
todo_items = ["Buy milk", "Write notes", "Walk the dog"]
with open("scratch/todo.txt", "w") as file:
    for item in todo_items:
        file.write(item + "\n")

# Exercise 2
with open("scratch/todo.txt", "r") as file:
    for line_number, line in enumerate(file, start=1):
        print(f"{line_number}. {line.strip()}")

# Exercise 3
def safe_get_item(items, index):
    try:
        return items[index]
    except IndexError:
        return None

print(safe_get_item([1, 2, 3], 1))
print(safe_get_item([1, 2, 3], 10))